# Sistema de Previsão de Quantidade de Propostas (TPOT AutoML)

## Objetivo
Este notebook implementa um pipeline de **Machine Learning Automático (AutoML)** usando TPOT para prever a quantidade de propostas em análise (etapa 16) no próximo dia útil.

## Fluxo Principal
1. **Verificação de dia útil**: Executa apenas em dias úteis (segunda a sexta, excluindo feriados brasileiros)
2. **Coleta de dados**: Extrai dados dos últimos 55 dias do ClickHouse
3. **Preparação**: Agrega dados diários e cria features com histórico (lags)
4. **Treinamento**: TPOT testa automaticamente vários modelos de regressão
5. **Validação**: Aceita modelos com MAPE (erro) entre 1-5%
6. **Previsão**: Faz previsão para o próximo dia
7. **Persistência**: Salva resultado e modelo em arquivos

## Critério de Aceitação
- **MAPE ideal**: 1% a 5% (erro percentual médio absoluto)
- **Tentativas**: Máximo 4 retreinamentos
- **Métrica**: Quanto menor o MAPE, melhor a previsão

## Arquivos
- `modelo_treinado-YYYY-MM-DD.pkl`: Modelo salvo por data
- `backup_qtd.xlsx`: Histórico de previsões realizadas


In [ ]:
# ==================== IMPORTAÇÕES ====================
# Bibliotecas para manipulação de dados
import pandas as pd  # Para trabalhar com dataframes e séries de dados
import joblib  # Para salvar e carregar modelos treinados (pickle)

# Bibliotecas para manipulação de arquivos e sistema operacional
import glob  # Para buscar arquivos com padrões (wildcards)
import os  # Para operações do sistema operacional

# Bibliotecas para data e hora
import datetime as dt  # Módulo de datetime
from datetime import datetime, timedelta  # Classes específicas de data/hora

# Bibliotecas de machine learning
from workadays import workdays as wd  # Para verificar dias úteis e feriados brasileiros
from tpot import TPOTRegressor  # AutoML - treina automaticamente diferentes modelos de regressão
from sklearn.metrics import mean_absolute_percentage_error  # Métrica MAPE para avaliar precisão do modelo

# Biblioteca para conexão com banco de dados
import clickhouse_connect  # Cliente para conectar ao ClickHouse (banco de dados)


c:\Users\daniel.ramazzotte\AppData\Local\anaconda3\Lib\site-packages\stopit\__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
# ==================== CONEXÃO COM BANCO DE DADOS ====================
# Configurar a conexão com o ClickHouse
client = clickhouse_connect.get_client(
        host='10.101.150.150',  # IP do servidor ClickHouse
        port=8123,  # Porta padrão do ClickHouse
        username='daniel_ramazzotte',  # Usuário do banco
        password='O9pLTAv*yVz0lGP^#M'  # Senha criptografada
    )

# ==================== QUERY SQL ====================
# Busca dados dos últimos 55 dias de propostas em análise
# Agrega por intervalos de 15 minutos (Intervalo)
# Calcula a soma do valor (SUM) e quantidade de propostas (COUNT)
query = """
SELECT
    toStartOfFifteenMinutes(ultimaalteracao) AS Intervalo,  -- Agrupa por intervalos de 15 min
    SUM(valor) AS valor,  -- Soma dos valores das propostas
    COUNT(propostaid) AS qntd  -- Quantidade de propostas
FROM
    crefaz.ft_proposta fp 
WHERE
    propostaetapaid = 16  -- Apenas propostas na etapa 16
    AND propostadecisaoid IS NULL  -- Que ainda não têm decisão
    AND toDate(ultimaalteracao) BETWEEN toDate(today() - INTERVAL 55 DAY) AND toDate(today() - INTERVAL 1 DAY)  -- Últimos 55 dias (excluindo hoje)
GROUP BY
    Intervalo  -- Agrupa os resultados por intervalo de tempo
"""


In [ ]:
# ==================== VERIFICAÇÃO DE DIA ÚTIL ====================
# Verifica se hoje é um dia útil (segunda a sexta) no Brasil
# Se for fim de semana ou feriado, o script não executa
if wd.is_workday(datetime.today(),country='BR'):
    continue  # Se for dia útil, continua com o resto do código

# Nota: Essa célula serve apenas como filtro/gate para execução dos blocos seguintes
# Se hoje não for dia útil, o código não prossegue


In [ ]:
if wd.is_workday(datetime.today(),country='BR'):

    # ==================== EXECUÇÃO DA QUERY ====================
    # Executa a query no ClickHouse e carrega o resultado em um DataFrame
    df = client.query_df(query)

    # ==================== PROCESSAMENTO INICIAL DOS DADOS ====================
    # Converte a coluna Intervalo para string (necessário para processamento posterior)
    df['Intervalo'] = df['Intervalo'].astype(str)  # Garante que a coluna seja string

    # Extrai a data e hora completa da coluna Intervalo (remove o timezone)
    df['data_hora'] = pd.to_datetime(df['Intervalo'].str.split('+').str[0])
    # Extrai apenas a data (sem hora)
    df['data'] = df['data_hora'].dt.date
    # Converte a data para datetime (necessário para comparações)
    df['data'] = pd.to_datetime(df['data'])

    # ==================== DEFINIÇÃO DA DATA DE REFERÊNCIA ====================
    # A data de referência é o último dia útil (não é sábado/domingo/feriado)
    hj = datetime.today()  # Data de hoje
    data_ontem = hj - timedelta(days=1)  # Data de ontem
    
    # Loop para encontrar o último dia útil (excluindo fins de semana e feriados)
    valida_data = False  # Flag para controlar o loop
    dias = 0  # Contador de dias para voltar
    while valida_data == False:
        # Calcula a data de referência subtraindo dias sucessivamente
        data_referencia = data_ontem - timedelta(days=dias)
        # Verifica se é feriado brasileiro
        feriado = wd.is_holiday(data_referencia,country = 'BR')
        # Obtém o dia da semana (0=segunda, 6=domingo)
        dia = data_referencia.weekday()
        # Se não for feriado E não for domingo (6), encontrou um dia útil válido
        if (feriado == False) & ( dia != 6):
            valida_data = True
        dias += 1

    # Define o período de treinamento (últimos 45 dias)
    data_min = data_referencia - timedelta(days=45)
    print('#####################################')
    print(f'Data de Referência: {data_referencia}')  # Data final do período
    print(f'Data Mínima: {data_min}')  # Data inicial do período
    print('#####################################')

    # ==================== PREPARAÇÃO DOS DADOS PARA TREINAMENTO ====================
    # Filtra o DataFrame para os últimos 45 dias
    df_agrupado = df[(df['data'] <= data_referencia) & (df['data'] >= data_min)]
    # Agrupa por dia e soma as quantidades e valores
    df_agrupado = df_agrupado.groupby('data').agg({'qntd': 'sum', 'valor': 'sum'}).reset_index()
    # Converte a quantidade para inteiro
    df_agrupado['qntd'] = df_agrupado['qntd'].astype(int)
    
    # ==================== CRIAÇÃO DE FEATURES (CARACTERÍSTICAS) ====================
    # Adiciona o dia da semana (0=segunda, 6=domingo) - informação útil para previsão
    df_agrupado['dia_semana'] = df_agrupado['data'].dt.day_of_week
    
    # Cria features de defasagem (lag features) - valores dos dias anteriores
    # lag_0: valor do dia anterior (pode ajudar a prever quantidade)
    df_agrupado['lag_0'] = df_agrupado['valor'].shift(1)
    # lag_1 até lag_6: quantidade dos 6 dias anteriores (histórico de 6 dias)
    df_agrupado['lag_1'] = df_agrupado['qntd'].shift(1)  # 1 dia atrás
    df_agrupado['lag_2'] = df_agrupado['qntd'].shift(2)  # 2 dias atrás
    df_agrupado['lag_3'] = df_agrupado['qntd'].shift(3)  # 3 dias atrás
    df_agrupado['lag_4'] = df_agrupado['qntd'].shift(4)  # 4 dias atrás
    df_agrupado['lag_5'] = df_agrupado['qntd'].shift(5)  # 5 dias atrás
    df_agrupado['lag_6'] = df_agrupado['qntd'].shift(6)  # 6 dias atrás
    
    # Remove linhas com valores nulos (primeiras 6 linhas terão NaN nos lags)
    df_agrupado.dropna(inplace=True)
    # Remove as colunas de data e valor (não são necessárias para o modelo)
    df_agrupado.drop(columns=['data', 'valor'], inplace=True)
    # Reseta o índice após as remoções
    df_agrupado.reset_index(drop=True, inplace=True)

    # ==================== SEPARAÇÃO DE TREINO E TESTE ====================
    # Separa os dados: treinamento com todos os dias menos o último, teste com o último dia
    train = df_agrupado[:-1]  # Todos os dias exceto o último
    teste = df_agrupado[-1:]  # Apenas o último dia
    # Separa features (X) de variável alvo (y) para treinamento
    X_train, y_train = train.drop(columns='qntd'), train['qntd']
    # Separa features (X) de variável alvo (y) para teste
    X_teste, y_teste = teste.drop(columns='qntd'), teste['qntd']

    # Inicializa contador para controlar quantas vezes treinar o modelo
    contador = 1


#####################################
2026-04-21 16:16:44.500786
2026-03-07 16:16:44.500786
#####################################


In [ ]:
if wd.is_workday(datetime.today(),country='BR'):

    # ==================== TREINAMENTO AUTOMÁTICO DO MODELO ====================
    # Loop para encontrar um modelo com MAPE (erro médio) aceitável
    # MAPE: Mean Absolute Percentage Error (quanto menor, melhor - ideal entre 1-5%)
    while True:
        # TPOT é uma ferramenta AutoML que testa automaticamente vários modelos de regressão
        # e encontra o melhor pipeline (sequência de transformações + modelo)
        model = TPOTRegressor(
            generations=60,  # 60 gerações de evolução genética
            population_size=60,  # 60 modelos por geração
            verbose=2,  # Modo verbose (mostra progresso)
            n_jobs=1,  # Usa 1 processador (pode ser alterado para -1 para usar todos)
            early_stop=8  # Para se não melhorar por 8 gerações
        )
        
        # Treina o modelo com os dados de treinamento
        model.fit(X_train, y_train)
        
        # Faz previsão nos dados de teste
        y_pred = model.predict(X_teste)
        
        # Calcula o MAPE (erro percentual médio absoluto) em porcentagem
        mape = mean_absolute_percentage_error(y_teste, y_pred) * 100

        # Exibe as métricas
        print('------------------------')
        print('METRICAS')
        print(f'MAPE: {mape}')
        print('------------------------')
        print(f'Contador: {contador}')
        
        # Lógica de aceitação do modelo:
        # Se contador <= 4 (até 4 tentativas):
        #   - Se MAPE > 5% ou MAPE < 1%: modelo não é bom, tenta novamente
        #   - Se 1% <= MAPE <= 5%: modelo aceitável, salva e encerra
        # Se contador > 4: atingiu limite de tentativas, usa o modelo mesmo que não seja ideal
        
        if (contador <= 4 and (mape > 5 or mape < 1)):
            # Modelo não atende ao critério (fora do intervalo 1-5%), treina novamente
            contador += 1
            continue
        elif contador <=4 and 1 <= mape <= 5:
            # Modelo com MAPE aceitável (entre 1-5%), salva e encerra o treinamento
            # Salva o pipeline do modelo treinado em um arquivo .pkl para posterior uso
            joblib.dump(model.fitted_pipeline_, f'C:\\Users\\daniel.ramazzotte\\OneDrive - Crefaz\\Área de Trabalho\\Projecoes\\Modelos_qtd\\modelo_treinado-{data_referencia.date()}.pkl')
            break
        elif contador > 4:
            # Atingiu 4 tentativas sem sucesso, encerra mesmo com modelo não ideal
            break


In [ ]:
if wd.is_workday(datetime.today(),country='BR'):

    # ==================== CARREGAMENTO DO MODELO MAIS RECENTE ====================
    # Busca todos os arquivos .pkl (modelos) na pasta de modelos
    arquivos = glob.glob(f'C:\\Users\\daniel.ramazzotte\\OneDrive - Crefaz\\Área de Trabalho\\Projecoes\Modelos_qtd\*.pkl')
    # Ordena os arquivos pelo tempo de modificação (mais recente primeiro)
    # -os.stat(t).st_mtime retorna o tempo de modificação em ordem decrescente
    arquivos_dec = sorted(arquivos, key=lambda t: -os.stat(t).st_mtime)
    # Carrega o modelo mais recente em memória
    modelo_treinado = joblib.load(arquivos_dec[0])

    # ==================== CARREGAMENTO DE DADOS PARA PREVISÃO ====================
    # Executa a mesma query novamente para dados atualizados
    plot = client.query_df(query)

    # Processa os dados da mesma forma que foi feito para treinamento
    plot['Intervalo'] = plot['Intervalo'].astype(str)  # Converte para string
    plot['data_hora'] = pd.to_datetime(plot['Intervalo'].str.split('+').str[0])  # Extrai data/hora
    plot['data'] = pd.to_datetime(plot['data_hora'].dt.date)  # Extrai apenas a data

    # Filtra os dados para o período de 45 dias
    plot_agrupado = plot[(plot['data'] <= data_referencia) & (plot['data'] >= data_min)]
    # Agrupa por dia
    plot_agrupado = plot_agrupado.groupby('data').agg({'qntd': 'sum', 'valor': 'sum'}).reset_index()
    # Converte a quantidade para float
    plot_agrupado['qntd'] = plot_agrupado['qntd'].astype(float)

    # ==================== CRIAÇÃO DE FEATURES PARA PREVISÃO ====================
    # Ajusta o dia da semana somando os dias passados (para obter o dia correto da previsão)
    plot_agrupado['dia_semana'] = (plot_agrupado['data'] + pd.to_timedelta(dias, unit='d')).dt.day_of_week
    # lag_0: valor atual (para manter coerência com features de treinamento)
    plot_agrupado['lag_0'] = plot_agrupado['valor']
    # lag_1 até lag_6: quantidade dos dias (histórico)
    plot_agrupado['lag_1'] = plot_agrupado['qntd']
    plot_agrupado['lag_2'] = plot_agrupado['qntd'].shift(1)
    plot_agrupado['lag_3'] = plot_agrupado['qntd'].shift(2)
    plot_agrupado['lag_4'] = plot_agrupado['qntd'].shift(3)
    plot_agrupado['lag_5'] = plot_agrupado['qntd'].shift(4)
    plot_agrupado['lag_6'] = plot_agrupado['qntd'].shift(5)
    
    # Remove as colunas desnecessárias (mantém apenas as features)
    plot_agrupado.drop(columns=['data', 'qntd', 'valor'], inplace=True)
    # Reseta o índice
    plot_agrupado.reset_index(drop=True, inplace=True)
    
    # ==================== REALIZAÇÃO DA PREVISÃO ====================
    # Faz a previsão usando a última linha dos dados (última data disponível)
    y_pred_next = modelo_treinado.predict(plot_agrupado.tail(1))

    # ==================== ATUALIZAÇÃO DO ARQUIVO DE BACKUP ====================
    # Carrega o histórico de previsões anterior
    base = pd.read_excel(r'C:\Users\daniel.ramazzotte\OneDrive - Crefaz\Área de Trabalho\Projecoes\backup_qtd.xlsx')
    
    # Cria um dicionário com os dados da previsão atual
    infos = {
        'Dia Referência': data_referencia,  # Data para a qual foi feita a previsão
        'Valor Previsto': y_pred_next[0],  # Valor previsto do modelo
        'Map': mape,  # MAPE (precisão) do modelo utilizado
        'Modelo Utilizado': arquivos_dec[0].split('\\')[-1].split('.pkl')[0],  # Nome do arquivo do modelo
        'Data previsao' : hj  # Data em que a previsão foi realizada
    }
    
    # Converte o dicionário em um DataFrame
    df_infos = pd.DataFrame([infos])
    # Concatena o novo registro com o histórico
    final = pd.concat([base, df_infos])
    # Salva o arquivo atualizado no Excel
    # final.to_excel(r'M:\03-Relatorios\Projecoes\backup_qtd.xlsx', index=False)  # Descomente para salvar
